### Generate data AWGN channel

In [12]:
import numpy as np
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import pickle
np.random.seed(42)
tf.random.set_seed(42)
plt.style.use('ggplot')
import sys
import numpy

### SETUP COPULA
sys.path.append("D:/NPC")
sys.path.append("D:/Houman/TEST_FLO")
#sys.path.append("C:/Users/alessandro/Documents/GitHub/NPC")
import tensorflow as tf
gpu_devices = tf.config.experimental.list_physical_devices('GPU')
device = gpu_devices[0]
# tf.config.experimental.set_memory_growth(device, True)
import tensorflow_probability as tfp
tfd = tfp.distributions
tfb = tfp.bijectors
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import sys
from classes.objects import *
from vine_tree.tree_op import *

from scipy import stats
import pickle

###########

from param.generate_rvine import *
from param.margin_fit import *
from param.margin_op import *
from param.copula_fit import *
from param.cond_copula import *
from pre_proc.preparation import prep_cop
from pred.prediction import*
from sampling.vine_sample import *
from info.info_estimation import vine_entropy

gpu_available = tf.test.is_gpu_available('GPU')

gpu_available = tf.config.list_physical_devices('GPU')
print(gpu_available)

print(tf.test.is_gpu_available('GPU:0'))
tf.test.gpu_device_name()
#from tensorflow.python.client import device_lib
#print(device_lib.list_local_devices())

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
True


'/device:GPU:0'

In [13]:
def sample_AWGN_channel(batch_size, dim, SIGNAL_NOISE = 0.5, SIGNAL_POWER = 2):
    """Simple additive white Gaussian noise channel"""
    x_sample = tf.random.normal((batch_size, dim), stddev = np.sqrt(SIGNAL_POWER))
    y_sample = x_sample + tf.random.normal((batch_size, dim), stddev = np.sqrt(SIGNAL_NOISE))   
    
    return tf.cast(x_sample, tf.float32), tf.cast(y_sample, tf.float32)

In [14]:
### THIS IS TO GENERATE THE DATA BUT IF YOU ALREADY LOADED IS NOT NECESSARY
dim = 10
batch_size = 1000
var1, var2 = sample_AWGN_channel(batch_size, dim)

#var1=var1[0:5,0:3]
#var2=var2[0:5,0:3]

#var1 = np.array(np.array(var1).transpose(1,0))
#var2 = np.array(np.array(var2).transpose(1,0))
#print(var1)
#print("")
#print(var2)
shuffle = False
if shuffle:
    var1=tf.random.shuffle(var1,seed=1)
    var2=tf.random.shuffle(var2,seed=100)
    
data = np.array([var1,var2]).transpose(1,0,2)
data = data.reshape(data.shape[0],-1)
print(data.shape)


(1000, 20)


### Fit copula var1-var2

In [10]:
################################ -  DEFINE THE VINE FOR FITTING - ####################################

### When defining the vine object
### If you use "r-vine", you have to add 'method' and 'r_matrix'
### E.g. using:
### r_matrix, ind_vine, nodes, matrix_edges = prepare_vine(vine_type, dim)
### and:
### vine = vine_obj_bin(vine_type, families, vine_total_dim, margin_vine, knots, method, r_matrix)

vine_type = "d-vine"
method = 'matrix' #'matrix' 'optimal'
families = "kercop"
knots = 50

vine_total_dim = data.shape[1]

r_matrix, ind_vine, nodes, E = random_r_matrix_gen(vine_total_dim)
#print(r_matrix)

## Define the margins
margin_vine = []
for i in range(0,vine_total_dim,1):
    mar_p = margin_obj('norm', [0,1], True)
    margin_vine.append(mar_p)
    
if vine_type == "r-vine":
    vine = vine_obj_bin(vine_type, families, vine_total_dim, margin_vine, knots, method, r_matrix)
else:
    vine = vine_obj_bin(vine_type, families, vine_total_dim, margin_vine, knots)

######### IF YOU WANT TO LOAD THE SAVED VINE --> Put load_pickle = True
load_pickle = False

if load_pickle:
    pickle_in = open("awgn_vine_16","rb")
    dict_save = pickle.load(pickle_in)
    # print(dict_save.keys())
    vine_copulas = dict_save["vine_copulas"]
    data = dict_save["data"]
    r_matrix = dict_save["r_matrix"]
    vine_depth = 2 * dim +1

    ## Here you load the copulas in the vine
    vine.copulas = vine_copulas
    vine.r_matrix = r_matrix
    vine.vine_depth = vine_depth

nodes:
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]
edges:
['(1,2)', '(2,3)', '(3,4)', '(4,5)', '(5,6)', '(6,7)', '(7,8)', '(8,9)', '(9,10)', '(10,11)', '(11,12)', '(12,13)', '(13,14)', '(14,15)', '(15,16)', '(16,17)', '(17,18)', '(18,19)', '(19,20)']
['(1,3|2)', '(2,4|3)', '(3,5|4)', '(4,6|5)', '(5,7|6)', '(6,8|7)', '(7,9|8)', '(8,10|9)', '(9,11|10)', '(10,12|11)', '(11,13|12)', '(12,14|13)', '(13,15|14)', '(14,16|15)', '(15,17|16)', '(16,18|17)', '(17,19|18)', '(18,20|19)']
['(1,4|2,3)', '(2,5|3,4)', '(3,6|4,5)', '(4,7|5,6)', '(5,8|6,7)', '(6,9|7,8)', '(7,10|8,9)', '(8,11|9,10)', '(9,12|10,11)', '(10,13|11,12)', '(11,14|12,13)', '(12,15|13,14)', '(13,16|14,15)', '(14,17|15,16)', '(15,18|16,17)', '(16,19|17,18)', '(17,20|18,19)']
['(1,5|2,3,4)', '(2,6|3,4,5)', '(3,7|4,5,6)', '(4,8|5,6,7)', '(5,9|6,7,8)', '(6,10|7,8,9)', '(7,11|8,9,10)', '(8,12|9,10,11)', '(9,13|10,11,12)', '(10,14|11,12,13)', '(11,15|12,13,14)', '(12,16|13,14,15)', '(13,17|14,15,16)', '(14,18|15,16,17

In [11]:
print(dim)

10


In [6]:
################################ -  VINE FITTING - ####################################
### Vine fitting instructions

# General parameters:
# - parallel: True or False           (Fit in parallel each level)
# - binning: True or False            (It can be True only if parallel is False)
# - param: True or False              (Parametric or Non-parametric)
# - vine_depth: any                   (Max level of vine to fit)
# - Fitted: True or False             (If the vine was already fitted, recompute some needed variables)

# Parametric parameters:
# - param_families: ["ind","gaussian","student","clayton","claytonrot90"]   (Decide which parametric families to fit)

# Non-parametric parameters:
# - opt_method: 'LL1' or 'LL2'

# Binning parameters:
# - n_bin: any                        (Number of bins)

### Parameters

param = False
binning = False
n_bins = 3
parallel = True

if binning:
    parallel = False

### Make data divisible for bins and k-fold

x = np.array(data,np.float32)

if binning == True:
    if param == False:
        exc = x.shape[0] % (n_bins*5) #tf.math.floormod(tf.shape(x)[0],n_bin*5)
    else:
        exc = x.shape[0] % n_bins #tf.math.floormod(tf.shape(x)[0],n_bin)
    x = x[:x.shape[0]-exc,:]
else:
    if param == False:
        exc = x.shape[0] % 5 #tf.math.floormod(tf.shape(x)[0],5)
        x = x[:x.shape[0]-exc,:]

### Prepare copula

sort_n = 'rand'
e = prep_cop(x, vine, sort_n)
print(e)

### FITTING
# Add parameters in a dictionary
vine_depth_fit = 2*dim+1

gen_dict = {'parallel':parallel, 'binning':binning, 'param':param, 'vine_depth':vine_depth_fit, 'fitted':False}  #vine_depth
par_dict = {'param_families':["ind","gaussian"]}  #["ind","gaussian","student","clayton","claytonrot90"]
npc_dict = {'opt_method':'LL1','batch_paral':3}
bin_dict = {'n_bin':n_bins}

save_vine = False

vine.fit(x,gen_dict,npc_dict,par_dict,bin_dict)

if save_vine:
    dict_save = {'vine_copulas': vine.copulas, 'r_matrix': vine.r_matrix, 'data': x}
    pickle_out = open("awgn_vine_16_2000","wb")
    pickle.dump(dict_save,pickle_out)
    pickle_out.close()

[[ 0.46311042 -1.1916528   0.45174745 ...  0.76228225  2.103735
  -0.5722866 ]
 [-2.1306705  -0.37412652 -0.84460235 ... -3.800331    0.7536462
   1.156131  ]
 [-0.31906855 -1.0771277  -2.6752875  ...  0.6515485   0.02151
  -0.41095397]
 ...
 [-0.30268982  0.6711062  -0.34345743 ...  1.2861409   0.33366507
   1.5376029 ]
 [ 0.35350353 -0.69038266  2.5832253  ...  0.5570228  -2.634469
   2.979889  ]
 [ 0.796826    0.31742638 -1.095095   ... -0.29608488 -2.416471
   4.106168  ]]
Instructions for updating:
This op will be removed after the deprecation date. Please switch to tf.sets.difference().
time_fit: 13.720729897946171
time_fit: 2.8173780781780238
time_fit: 2.955711872894618
time_fit: 4.452793960752356
time_fit: 3.226343325132113
time_fit: 3.4920216984917474
time_fit: 2.828736815584591
time_fit: 5.889051841852336
time_fit: 1.9817391764153314
time_fit: 4.020555712806029
time_fit: 0.7956111565582233
time_fit: 2.971789472374553
time_fit: 2.8330718482598343
time_fit: 2.104368625523705
ti

KeyboardInterrupt: 

### Fit copula v2

In [ ]:
################################ -  DEFINE THE VINE FOR FITTING - ####################################

### When defining the vine object
### If you use "r-vine", you have to add 'method' and 'r_matrix'
### E.g. using:
### r_matrix, ind_vine, nodes, matrix_edges = prepare_vine(vine_type, dim)
### and:
### vine = vine_obj_bin(vine_type, families, vine_total_dim, margin_vine, knots, method, r_matrix)

#vine_type = "d-vine"
#method = 'matrix' #'matrix' 'optimal'
#families = "kercop"
#knots = 50

var_x2 = data[:,:dim]

vine_total_dim_x2 = var_x2.shape[1]

r_matrix_x2, ind_vine_x2, nodes_x2, E_x2 = random_r_matrix_gen(vine_total_dim_x2)
#print(r_matrix_x2)

## Define the margins
margin_vine_x2 = []
for i in range(0,vine_total_dim_x2,1):
    mar_p = margin_obj('norm', [0,1], True)
    margin_vine_x2.append(mar_p)
    

if vine_type == "r-vine":
    vine_x2 = vine_obj_bin(vine_type, families, vine_total_dim_x2, margin_vine_x2, knots, method, r_matrix_x2)
else:
    vine_x2 = vine_obj_bin(vine_type, families, vine_total_dim_x2, margin_vine_x2, knots)

######### IF YOU WANT TO LOAD THE SAVED VINE --> Put load_pickle = True
load_pickle = False

if load_pickle:
    pickle_in = open("awgn_vine_x2_8","rb")
    dict_save = pickle.load(pickle_in)
    # print(dict_save.keys())
    vine_copulas_x2 = dict_save["vine_copulas"]
    # sample = dict_save["data"]
    r_matrix_x2 = dict_save["r_matrix"]
    vine_depth_x2 = 8

    ## Here you load the copulas in the vine
    vine_x2.copulas = vine_copulas_x2
    vine_x2.r_matrix = r_matrix_x2
    vine_x2.vine_depth = vine_depth_x2

In [ ]:
################################ -  VINE FITTING - ####################################
### Vine fitting instructions

# General parameters:
# - parallel: True or False           (Fit in parallel each level)
# - binning: True or False            (It can be True only if parallel is False)
# - param: True or False              (Parametric or Non-parametric)
# - vine_depth: any                   (Max level of vine to fit)
# - Fitted: True or False             (If the vine was already fitted, recompute some needed variables)

# Parametric parameters:
# - param_families: ["ind","gaussian","student","clayton","claytonrot90"]   (Decide which parametric families to fit)

# Non-parametric parameters:
# - opt_method: 'LL1' or 'LL2'

# Binning parameters:
# - n_bin: any                        (Number of bins)

### Parameters

#param = False
#binning = True
#n_bins = 3
#parallel = True

if binning:
    parallel = False

### Make data divisible for bins and k-fold

x = np.array(var_x2,np.float32)

if binning == True:
    if param == False:
        exc = x.shape[0] % (n_bins*5) #tf.math.floormod(tf.shape(x)[0],n_bin*5)
    else:
        exc = x.shape[0] % n_bins #tf.math.floormod(tf.shape(x)[0],n_bin)
    x = x[:x.shape[0]-exc,:]
else:
    if param == False:
        exc = x.shape[0] % 5 #tf.math.floormod(tf.shape(x)[0],5)
        x = x[:x.shape[0]-exc,:]

### Prepare copula

sort_n = 'rand'
e = prep_cop(x, vine_x2, sort_n)
print(e)

### FITTING
# Add parameters in a dictionary
vine_depth_fit = dim+1

gen_dict = {'parallel':parallel, 'binning':binning, 'param':param, 'vine_depth':vine_depth_fit, 'fitted':False}  #vine_depth
par_dict = {'param_families':["ind","gaussian"]}  #["ind","gaussian","student","clayton","claytonrot90"]
npc_dict = {'opt_method':'LL1','batch_paral':3}
bin_dict = {'n_bin':n_bins}

save_vine = False

vine_x2.fit(x,gen_dict,npc_dict,par_dict,bin_dict)

if save_vine:
    dict_save = {'vine_copulas': vine.copulas, 'r_matrix': vine.r_matrix, 'data': x}
    pickle_out = open("awgn_8_2000","wb")
    pickle.dump(dict_save,pickle_out)
    pickle_out.close()

In [ ]:
#Fit copula v3
################################ -  DEFINE THE VINE FOR FITTING - ####################################

### When defining the vine object
### If you use "r-vine", you have to add 'method' and 'r_matrix'
### E.g. using:
### r_matrix, ind_vine, nodes, matrix_edges = prepare_vine(vine_type, dim)
### and:
### vine = vine_obj_bin(vine_type, families, vine_total_dim, margin_vine, knots, method, r_matrix)

#vine_type = "d-vine"
#method = 'matrix' #'matrix' 'optimal'
#families = "kercop"
#knots = 50

var_x3 = data[:,dim:]

vine_total_dim_x3 = var_x3.shape[1]

r_matrix_x3, ind_vine_x3, nodes_x3, E_x3 = random_r_matrix_gen(vine_total_dim_x3)
#print(r_matrix_x3)

## Define the margins
margin_vine_x3 = []
for i in range(0,vine_total_dim_x3,1):
    mar_p = margin_obj('norm', [0,1], True)
    margin_vine_x3.append(mar_p)
    
if vine_type == "r-vine":
    vine_x3 = vine_obj_bin(vine_type, families, vine_total_dim_x3, margin_vine_x3, knots, method, r_matrix_x3)
else:
    vine_x3 = vine_obj_bin(vine_type, families, vine_total_dim_x3, margin_vine_x3, knots)

######### IF YOU WANT TO LOAD THE SAVED VINE --> Put load_pickle = True
load_pickle = False

if load_pickle:
    pickle_in = open("awgn_vine_x2_8","rb")
    dict_save = pickle.load(pickle_in)
    # print(dict_save.keys())
    vine_copulas_x2 = dict_save["vine_copulas"]
    # sample = dict_save["data"]
    r_matrix_x2 = dict_save["r_matrix"]
    vine_depth_x2 = 8

    ## Here you load the copulas in the vine
    vine_x3.copulas = vine_copulas_x3
    vine_x3.r_matrix = r_matrix_x3
    vine_x3.vine_depth = vine_depth_x3

################################ -  VINE FITTING - ####################################
### Vine fitting instructions

# General parameters:
# - parallel: True or False           (Fit in parallel each level)
# - binning: True or False            (It can be True only if parallel is False)
# - param: True or False              (Parametric or Non-parametric)
# - vine_depth: any                   (Max level of vine to fit)
# - Fitted: True or False             (If the vine was already fitted, recompute some needed variables)

# Parametric parameters:
# - param_families: ["ind","gaussian","student","clayton","claytonrot90"]   (Decide which parametric families to fit)

# Non-parametric parameters:
# - opt_method: 'LL1' or 'LL2'

# Binning parameters:
# - n_bin: any                        (Number of bins)

### Parameters

#param = False
#binning = True
#n_bins = 3
#parallel = True

if binning:
    parallel = False

### Make data divisible for bins and k-fold

x = np.array(var_x3,np.float32)

if binning == True:
    if param == False:
        exc = x.shape[0] % (n_bins*5) #tf.math.floormod(tf.shape(x)[0],n_bin*5)
    else:
        exc = x.shape[0] % n_bins #tf.math.floormod(tf.shape(x)[0],n_bin)
    x = x[:x.shape[0]-exc,:]
else:
    if param == False:
        exc = x.shape[0] % 5 #tf.math.floormod(tf.shape(x)[0],5)
        x = x[:x.shape[0]-exc,:]

### Prepare copula

sort_n = 'rand'
e = prep_cop(x, vine_x3, sort_n)
print(e)

### FITTING
# Add parameters in a dictionary
vine_depth_fit = dim+1

gen_dict = {'parallel':parallel, 'binning':binning, 'param':param, 'vine_depth':vine_depth_fit, 'fitted':False}  #vine_depth
par_dict = {'param_families':["ind","gaussian"]}  #["ind","gaussian","student","clayton","claytonrot90"]
npc_dict = {'opt_method':'LL1','batch_paral':3}
bin_dict = {'n_bin':n_bins}

save_vine = False

vine_x3.fit(x,gen_dict,npc_dict,par_dict,bin_dict)

if save_vine:
    dict_save = {'vine_copulas': vine.copulas, 'r_matrix': vine.r_matrix, 'data': x}
    pickle_out = open("awgn_8_2000","wb")
    pickle.dump(dict_save,pickle_out)
    pickle_out.close()

In [ ]:
def cond_vine_entropy(vine,vine_f2,vine_f3,info_dict):
    alpha = info_dict['alpha']
    cases = info_dict['cases'] #number of samples in each iteration 
    max_iter = info_dict['iterations']
    d = vine.n_cop
    d_f2 = vine_f2.n_cop

    norm_dis = tfd.Normal(loc=0., scale=1.) 
    conf = norm_dis.quantile(1-alpha)
    tim = 0  #Add as parameter if you want to change it

    mo = 0 
    varsum1 = 0 
    infoc1 = 0
    cond_entr = 0
    entr_f2 = 0
    entr_f3 = 0
    stderr1 = 1e+6
    stderr2 = 1e+6 
    stderr3 = 1e+6 
    stderr_tot = 1e+6
    erreps = 1e-3
    
    MI_XY=numpy.zeros(max_iter,numpy.float)
    MI_SING=numpy.zeros(max_iter,numpy.float)
    EN_XY=numpy.zeros(max_iter,numpy.float)
    EN_X1=numpy.zeros(max_iter,numpy.float)
    EN_X2=numpy.zeros(max_iter,numpy.float)

    mag = tf.math.reduce_max(vine.grid_u.ex)
    mig = tf.math.reduce_min(vine.grid_u.ex)

    mag_f2 = tf.math.reduce_max(vine_f2.grid_u.ex)
    mig_f2 = tf.math.reduce_min(vine_f2.grid_u.ex)

    while ((stderr1 >= erreps) | (stderr2 >= erreps) | (stderr_tot >= erreps) ) & (mo < max_iter):
        mo = mo+1
        if vine.param == False:

            ## Sample from joint copula and compute prob.
            w = tf.random.uniform([cases,d], minval=0, maxval=1, dtype=vine.data_x.dtype)
            w = (mag-mig)*(w-tf.math.reduce_min(w))/(tf.math.reduce_max(w)-tf.math.reduce_min(w))+mig
            
            sample , u, po, op = vine_copula_sample(vine,cases)
            
            #var1, var2 = sample_AWGN_channel(cases, d_f2)
            #data = np.array([var1,var2]).transpose(1,0,2)
            #data = data.reshape(data.shape[0],-1)
            sample = data


            p, p_copula, log_marg_f= vine.evaluation(sample)

            ## Sample from var2 copula and compute prob.
    
            sample_f2 = sample[:,:d_f2]
            #sample_f2,u_f2, po, op = vine_copula_sample(vine_f2,cases)
            
            p_f2, p_copula_f2, log_marg_f2 = vine_f2.evaluation(sample_f2)

            ## Sample from var3 copula and compute prob.
    
            sample_f3 = sample[:,d_f2:]
            #sample_f3,u_f3, po, op = vine_copula_sample(vine_f3,cases)
            
            p_f3, p_copula_f3, log_marg_f3 = vine_f3.evaluation(sample_f3)

            ## Compute cond entr.

            p_cond = np.exp(np.log(p_copula.numpy()))
            #- np.log(p_f2.numpy()))
            
            log2_cond = np.log2(p_cond)
            log2_cond[p_cond == 0] = 0 

            cond_12=np.mean(log2_cond);
            cond_entr = cond_entr + ( np.mean(log2_cond) - cond_entr) / mo  #tf.math.reduce_mean

            varsum1 = varsum1 + np.sum((log2_cond - cond_entr)**2)
            stderr1 = conf * np.sqrt(varsum1 / (mo * cases * (mo * cases - 1)))
            
            ## Compute entr.
            log2_f2 = np.log2(p_copula_f2.numpy())
            log2_f2[p_f2 == 0] = 0 
            entr_2=np.mean(log2_f2);
            entr_f2 = entr_f2 + ( np.mean(log2_f2) - entr_f2) / mo

            ## Compute entr.            
            log2_f3 = np.log2(p_copula_f3.numpy())
            log2_f3[p_f3 == 0] = 0             
            entr_3=np.mean(log2_f3);
            entr_f3 = entr_f3 + ( np.mean(log2_f3) - entr_f3) / mo
            
            print(mo, ' : MI: ', -(entr_f2 + entr_f3 - cond_entr))
            print(mo, ' : MI_SING: ', -(entr_2 + entr_3 - cond_12))

            MI_XY[mo-1] = -(entr_f2 + entr_f3 - cond_entr)
            MI_SING[mo-1] = -(entr_2 + entr_3 - cond_12)
            EN_X1[mo-1] = entr_f2
            EN_X2[mo-1] = entr_f3 
            EN_XY[mo-1] = cond_entr
        else:
            sample = vine_cop_par_sample(vine,cases)

            # Compute pdf of samples
            p, pcop = vine.evaluation(sample)
            
            log2pp = np.log2(pcop.numpy())
            log2pp[pcop == 0] = 0 

            infoc1 = infoc1 + ( np.mean(log2pp) - infoc1) / mo
            varsum1 = varsum1 + np.sum((log2pp - infoc1)**2)
            stderr1 = conf * np.sqrt(varsum1 / (mo * cases * (mo * cases - 1)))
            
    return EN_XY, EN_X1, EN_X2, MI_XY, MI_SING

In [ ]:
info_dict = {'cases':2000, 'iterations':20, 'alpha': 0.05}
EN_XY, EN_X1, EN_X2, MI_XY, MI_SING = cond_vine_entropy(vine,vine_x2,vine_x3,info_dict)
print('MI: ', MI_XY[-1])
plt.plot(MI_XY)

In [ ]:
def theoretic_mutual_information_AWGN(power, noise, dim):
    return dim * 0.5 * np.log2(1 + power/noise)

th_mi = theoretic_mutual_information_AWGN(2, 0.5,dim)
print(th_mi)

### Example sample vine

In [ ]:
cases = 5000
sample, u, pf, ps = vine_copula_sample(vine,cases)

fig, axs = plt.subplots(vine.n_cop, vine.n_cop,figsize=(2*dim+1,2*dim+1))
for i in range(0,vine.n_cop,1):
    for j in range(i+1,vine.n_cop,1):
        axs[i,j].plot(data[:cases,i],data[:cases,j],'b.')    
        axs[i,j].plot(sample[:,i],sample[:,j],'r.')    
        #axs[i,j].plot(u[:,i],u[:,j],'r.')    
        #axs[i,j].set_title(str(i+1)+","+str(j+1))
for i in range(0,vine.n_cop,1):
        print(i+1,min(data[:cases,i]),min(sample[:,i]))        
        print(i+1,max(data[:cases,i]),max(sample[:,i]))

In [ ]:
cases = 2000
sample_x2, u, pf2, ps2 = vine_copula_sample(vine_x2,cases)

fig, axs = plt.subplots(vine_x2.n_cop, vine_x2.n_cop,figsize=(30,30))
for i in range(0,vine_x2.n_cop,1):
    for j in range(i+1,vine_x2.n_cop,1):
        axs[i,j].plot(data[:,vine_x2.n_cop+i],data[:,vine_x2.n_cop+j],'b.')    
        axs[i,j].plot(sample_x2[:,i],sample_x2[:,j],'r.')    
        axs[i,j].set_title(str(i+1)+","+str(j+1))

In [ ]:
cases = 2000
sample_x3, u, pf3, ps3 = vine_copula_sample(vine_x3,cases)

fig, axs = plt.subplots(vine_x3.n_cop, vine_x3.n_cop,figsize=(30,30))
for i in range(0,vine_x3.n_cop,1):
    for j in range(i+1,vine_x3.n_cop,1):
        axs[i,j].plot(data[:,vine_x3.n_cop+i],data[:,vine_x3.n_cop+j],'b.')    
        axs[i,j].plot(sample_x3[:,i],sample_x3[:,j],'r.')    
        axs[i,j].set_title(str(i+1)+","+str(j+1))